<a href="https://colab.research.google.com/github/angeruzzi/recommender_system_movielens/blob/main/05_collaborative_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaborative Filtering — MovieLens 100K

## 1. Introdução

Este notebook implementa sistemas de recomendação baseados em **Collaborative Filtering**, utilizando exclusivamente os padrões de interação entre usuários e filmes.

Diferentemente da abordagem Content-Based, os modelos colaborativos não dependem de atributos explícitos dos filmes, como gêneros. As recomendações são produzidas a partir de relações identificadas no comportamento coletivo dos usuários.

Serão avaliadas duas estratégias:

- **User-Based Collaborative Filtering**: identifica usuários com padrões de avaliação semelhantes e utiliza suas preferências para recomendar novos itens;
- **Item-Based Collaborative Filtering**: identifica filmes avaliados de maneira semelhante pelos usuários e utiliza essas relações para produzir recomendações.

Os modelos serão avaliados utilizando o mesmo split temporal e as mesmas métricas Top-K dos experimentos anteriores.

# 2. Preparação

In [21]:
#Imports

!wget -q -O utils.py \
    https://raw.githubusercontent.com/angeruzzi/recommender_system_movielens/main/utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

import utils

#Validação
assert utils.validate_metrics()
assert utils.validate_framework()

print("Framework carregado com sucesso.")

#Configuração experimental:
RELEVANCE_THRESHOLD = 4
TRAIN_RATIO = 0.8
K = 10
RANDOM_STATE = 42
N_NEIGHBORS = 30
MIN_COMMON_ITEMS = 3

#Carregamento
ratings, movies = utils.load_movielens_100k()

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['user_id'].nunique():,}")
print(f"Movies:  {ratings['item_id'].nunique():,}")

#Split
train, test = utils.temporal_split_by_user(
    ratings,
    train_ratio=TRAIN_RATIO
)

evaluation_data = utils.build_evaluation_data(
    train=train,
    test=test,
    relevance_threshold=RELEVANCE_THRESHOLD
)

train_catalog = evaluation_data["train_catalog"]
seen_items = evaluation_data["seen_items"]
ground_truth = evaluation_data["evaluable_ground_truth"]
evaluation_users = evaluation_data["evaluation_users"]

print(f"Train:              {len(train):,}")
print(f"Test:               {len(test):,}")
print(f"Evaluation users:   {len(evaluation_users):,}")
print(f"Train catalog:      {len(train_catalog):,}")

RESULTS_URL = "https://raw.githubusercontent.com/angeruzzi/recommender_system_movielens/main/model_results2.csv"
model_results = pd.read_csv(RESULTS_URL)

assert utils.validate_temporal_split(train, test).all()
print("Protocolo experimental validado.")

Framework carregado com sucesso.
Ratings: 100,000
Users:   943
Movies:  1,682
Train:              79,619
Test:               20,381
Evaluation users:   907
Train catalog:      1,611
Protocolo experimental validado.


## 3. Matriz Usuário × Item

Os modelos de Collaborative Filtering serão construídos a partir da matriz usuário-item formada apenas pelas interações do conjunto de treino.

As linhas representam usuários, as colunas representam filmes e as células contêm os ratings observados.

In [3]:
user_item_matrix = train.pivot(
    index="user_id",
    columns="item_id",
    values="rating"
)

user_item_matrix.shape

(943, 1611)

### 3.1 Centralização dos Ratings

Usuários podem apresentar diferentes padrões na utilização da escala de avaliações.

Para reduzir esse efeito, os ratings serão centralizados pela média de cada usuário:

\[
r'_{ui}=r_{ui}-\bar r_u
\]

Assim, os valores passam a representar desvios em relação ao comportamento habitual do próprio usuário.

In [6]:
user_means = user_item_matrix.mean(axis=1)

centered_matrix = user_item_matrix.sub(
    user_means,
    axis=0
)

#Para calcular similaridade por cosseno, podemos utilizar zero apenas como representação computacional da ausência de informação após centralização
centered_filled = centered_matrix.fillna(0)

centered_filled.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1660,1662,1663,1664,1670,1672,1673,1675,1676,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.405530,-0.594470,0.0,-0.59447,0.0,0.0,0.40553,-2.59447,0.0,-0.594470,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.285714,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,-1.714286,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.964286,-0.035714,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. User-Based Collaborative Filtering

O User-Based Collaborative Filtering procura usuários com padrões de preferência semelhantes.

Para um usuário \(u\), o modelo identifica os usuários mais similares e utiliza suas avaliações para estimar a preferência de \(u\) por itens ainda não avaliados.

### 4.1 Similaridade entre Usuários

A similaridade entre dois usuários será calculada considerando exclusivamente os itens avaliados por ambos (*co-rated items*).

Essa restrição evita interpretar a ausência de uma avaliação como um valor numérico e torna a comparação baseada apenas em evidências efetivamente compartilhadas pelos usuários.

Será utilizada a correlação de Pearson, que mede a similaridade entre os padrões relativos de avaliação dos dois usuários.

Também será exigido um número mínimo de itens coavaliados para reduzir similaridades pouco confiáveis.

In [29]:
def compute_user_similarity(
    user_item_matrix,
    min_common_items=3
):
    user_ids = user_item_matrix.index
    n_users = len(user_ids)

    similarity = np.zeros(
        (n_users, n_users),
        dtype=float
    )

    overlap = np.zeros(
        (n_users, n_users),
        dtype=int
    )

    for i in range(n_users):
        ratings_i = user_item_matrix.iloc[i]

        for j in range(i + 1, n_users):
            ratings_j = user_item_matrix.iloc[j]

            common = (
                ratings_i.notna()
                & ratings_j.notna()
            )

            n_common = common.sum()

            overlap[i, j] = n_common
            overlap[j, i] = n_common

            if n_common < min_common_items:
                continue

            x = ratings_i[common].values
            y = ratings_j[common].values

            # Pearson
            if np.std(x) == 0 or np.std(y) == 0:
                sim = 0.0
            else:
                sim = np.corrcoef(x, y)[0, 1]

            similarity[i, j] = sim
            similarity[j, i] = sim

    similarity_df = pd.DataFrame(
        similarity,
        index=user_ids,
        columns=user_ids
    )

    overlap_df = pd.DataFrame(
        overlap,
        index=user_ids,
        columns=user_ids
    )

    return similarity_df, overlap_df

In [30]:
# user_similarity_values = cosine_similarity(
#     centered_filled.values
# )

# user_similarity = pd.DataFrame(
#     user_similarity_values,
#     index=centered_filled.index,
#     columns=centered_filled.index
# )

# np.fill_diagonal(
#     user_similarity.values,
#     0
# )

# user_similarity.head()

In [8]:
user_similarity_pearson, user_overlap_pearson = (
    compute_user_similarity(
        user_item_matrix,
        min_common_items=MIN_COMMON_ITEMS
    )
)

### 4.2 Sobreposição entre Usuários

A similaridade entre dois usuários só é calculada quando existe uma quantidade mínima de itens avaliados por ambos.

A função utilizada na etapa anterior também retorna uma matriz de sobreposição, na qual cada célula representa o número de itens coavaliados por um par de usuários.

Pares com menos de `MIN_COMMON_ITEMS` interações em comum recebem similaridade igual a zero e não participam da formação da vizinhança.

In [46]:
user_overlap_pearson.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,0,12,3,2,55,61,108,26,3,58,...,60,8,29,9,23,8,34,5,18,60
2,12,0,7,5,5,24,13,6,4,13,...,11,10,24,13,20,10,11,5,6,7
3,3,7,0,8,0,6,5,5,1,5,...,2,1,12,6,7,2,11,3,7,0
4,2,5,8,0,2,4,5,4,1,2,...,3,1,6,5,5,1,6,2,6,3
5,55,5,0,2,0,24,76,15,3,26,...,42,6,10,3,11,5,18,3,11,40


In [47]:
overlap_values = user_overlap_pearson.values

positive_overlaps = overlap_values[
    overlap_values > 0
]

print(f"Média de itens em comum:   {positive_overlaps.mean():.2f}")
print(f"Mediana de itens em comum: {np.median(positive_overlaps):.2f}")
print(f"Máximo de itens em comum:  {positive_overlaps.max()}")

Média de itens em comum:   14.21
Mediana de itens em comum: 7.00
Máximo de itens em comum:  270


In [49]:
invalid_pairs = (
    (user_overlap_pearson < MIN_COMMON_ITEMS)
    & (user_similarity_pearson != 0)
)

print(
    "Pares abaixo do mínimo com similaridade diferente de zero:",
    invalid_pairs.sum().sum()
)

Pares abaixo do mínimo com similaridade diferente de zero: 0


### 4.3 Seleção dos Vizinhos

Para cada usuário, serão selecionados os usuários com maior similaridade de Pearson entre aqueles que possuem quantidade mínima de itens coavaliados.

Como pares com sobreposição insuficiente já receberam similaridade igual a zero na etapa anterior, eles são automaticamente excluídos da vizinhança.

Os vizinhos são ordenados da maior para a menor similaridade e os `N_NEIGHBORS` mais próximos são utilizados na predição.

In [50]:
def get_user_neighbors(
    user_id,
    similarity_matrix,
    n_neighbors=30
):
    similarities = (
        similarity_matrix
        .loc[user_id]
        .drop(index=user_id, errors="ignore")
    )

    # Remove pares sem similaridade válida
    similarities = similarities[
        similarities != 0
    ]

    # Mais similares primeiro
    similarities = similarities.sort_values(
        ascending=False
    )

    return similarities.head(n_neighbors)

In [51]:
user_id = evaluation_users[0]

neighbors = get_user_neighbors(
    user_id=user_id,
    similarity_matrix=user_similarity_pearson,
    n_neighbors=N_NEIGHBORS
)

neighbors.head(10)

,1
user_id,
139,1.000000
724,0.980196
570,0.970725
803,0.965824
905,0.963087
520,0.960769
772,0.944911
926,0.944911
572,0.944911


In [52]:
print(f"Usuário analisado: {user_id}")
print(f"Número de vizinhos encontrados: {len(neighbors)}")
print(f"Maior similaridade: {neighbors.max():.4f}")
print(f"Menor similaridade entre os selecionados: {neighbors.min():.4f}")

Usuário analisado: 1
Número de vizinhos encontrados: 30
Maior similaridade: 1.0000
Menor similaridade entre os selecionados: 0.7778


### 4.4 Predição de Scores

Para cada item candidato, o score será estimado a partir das avaliações dos usuários vizinhos que interagiram com esse item.

Como os ratings dos vizinhos foram centralizados em relação às suas próprias médias, cada avaliação representa um desvio em relação ao comportamento habitual daquele usuário.

Esses desvios são ponderados pela similaridade entre o vizinho e o usuário alvo. A média ponderada resultante é então adicionada à média histórica do usuário alvo, produzindo um rating estimado para o item.

In [54]:
def predict_user_based_scores(
    user_id,
    centered_matrix,
    user_means,
    similarity_matrix,
    train_catalog,
    seen_items,
    n_neighbors=30
):

    # Seleciona os vizinhos mais similares
    neighbors = get_user_neighbors(
        user_id=user_id,
        similarity_matrix=similarity_matrix,
        n_neighbors=n_neighbors
    )

    if neighbors.empty:
        return pd.Series(dtype=float)

    # Itens do catálogo de treino ainda não vistos pelo usuário
    candidates = utils.get_candidate_items(
        user_id=user_id,
        train_catalog=train_catalog,
        seen_items=seen_items
    )

    scores = {}

    # Calcula um score para cada item candidato
    for item_id in candidates:

        if item_id not in centered_matrix.columns:
            continue

        # Ratings centralizados dos vizinhos para o item
        neighbor_ratings = centered_matrix.loc[
            neighbors.index,
            item_id
        ]

        # Mantém apenas vizinhos que avaliaram o item
        available = neighbor_ratings.notna()

        if not available.any():
            continue

        item_neighbors = neighbor_ratings[available]

        # Similaridades dos vizinhos que avaliaram o item
        similarities = neighbors.loc[
            item_neighbors.index
        ]

        denominator = np.abs(similarities).sum()

        if denominator == 0:
            continue

        # Média ponderada dos desvios dos ratings
        weighted_deviation = (
            similarities * item_neighbors
        ).sum() / denominator

        # Retorna o score para a escala original do usuário
        predicted_rating = (
            user_means.loc[user_id]
            + weighted_deviation
        )

        scores[item_id] = predicted_rating

    return pd.Series(
        scores,
        dtype=float
    ).sort_values(ascending=False)

In [55]:
# Teste

user_id = evaluation_users[0]

user_scores = predict_user_based_scores(
    user_id=user_id,
    centered_matrix=centered_matrix,
    user_means=user_means,
    similarity_matrix=user_similarity_pearson,
    train_catalog=train_catalog,
    seen_items=seen_items,
    n_neighbors=N_NEIGHBORS
)

user_scores.head(10)

,0
287,6.049016
316,5.488588
902,5.488588
475,5.125720
298,5.125720
477,5.112989
297,4.829983
334,4.790825
285,4.746262
272,4.725401


### 4.5 Recomendações Top-K

Os itens candidatos serão ordenados pelo rating estimado. Os \(K\) itens com maior score formarão a lista final de recomendações do User-Based Collaborative Filtering.

In [56]:
user_cf_recommendations = {}

for user_id in evaluation_users:

    scores = predict_user_based_scores(
        user_id=user_id,
        centered_matrix=centered_matrix,
        user_means=user_means,
        similarity_matrix=user_similarity_pearson,
        train_catalog=train_catalog,
        seen_items=seen_items,
        n_neighbors=N_NEIGHBORS
    )

    user_cf_recommendations[user_id] = (
        scores
        .head(K)
        .index
        .tolist()
    )

In [58]:
#Validação do tamanho das listas
recommendation_lengths = pd.Series({
    user_id: len(items)
    for user_id, items in user_cf_recommendations.items()
})

recommendation_lengths.describe()

,0
count,907.0
mean,10.0
std,0.0
min,10.0
25%,10.0
50%,10.0
75%,10.0
max,10.0


### 4.6 Avaliação do User-Based CF

In [59]:
user_cf_results, user_cf_summary = (
    utils.evaluate_recommendations(
        recommendations=user_cf_recommendations,
        ground_truth=ground_truth,
        k=K
    )
)

user_cf_summary

{'Precision@10': np.float64(0.013671444321940463),
 'Recall@10': np.float64(0.01193955057281341),
 'NDCG@10': np.float64(0.015679018530965115),
 'n_users': 907}

In [79]:
comparison = utils.add_model_result(
    results_df=model_results,
    model_name="User-Based CF",
    summary=user_cf_summary,
    k=K
)

comparison

,model,Precision@10,Recall@10,NDCG@10,n_users
0,Random,0.007387,0.006949,0.008209,907
1,Popularity,0.077508,0.078646,0.098883,907
2,Content-Based,0.018964,0.018318,0.025018,907
3,User-Based CF,0.013671,0.011940,0.015679,907


### 4.7 Análise dos Resultados

O User-Based Collaborative Filtering apresentou desempenho superior ao Random Baseline nas três métricas avaliadas, indicando que a similaridade entre usuários contém informação útil para a geração das recomendações.

Entretanto, o modelo apresentou desempenho inferior ao Content-Based e, principalmente, ao Popularity Baseline.

A diferença em relação ao Popularity sugere que a popularidade global dos itens representa um sinal particularmente forte neste conjunto de dados e protocolo experimental.

O resultado também evidencia algumas limitações do User-Based CF baseado em vizinhança. A matriz usuário-item é esparsa, a similaridade entre usuários depende da existência de itens coavaliados e as estimativas para cada item são produzidas a partir de um subconjunto variável dos vizinhos disponíveis.

Apesar disso, o resultado representa uma melhora em relação à primeira implementação baseada em similaridade de cosseno com preenchimento dos valores ausentes. A utilização da correlação de Pearson calculada exclusivamente sobre itens coavaliados produziu uma vizinhança mais adequada ao problema de ratings explícitos.

A próxima etapa será avaliar o Item-Based Collaborative Filtering, no qual as relações de similaridade serão estabelecidas entre filmes em vez de usuários.

## 5. Item-Based Collaborative Filtering

O Item-Based Collaborative Filtering modela relações de similaridade entre itens a partir dos padrões de avaliação dos usuários.

Diferentemente do Content-Based Filtering, a similaridade não é determinada por atributos dos filmes, como gênero. Dois filmes são considerados semelhantes quando apresentam padrões de avaliação semelhantes entre os usuários que avaliaram ambos.

Para cada item candidato, o modelo utilizará os filmes já avaliados pelo usuário para estimar sua preferência, ponderando os ratings observados pela similaridade entre os itens.

Assim como no User-Based CF, a similaridade será calculada utilizando correlação de Pearson exclusivamente sobre observações compartilhadas.

In [108]:
MIN_COMMON_USERS = 3
SIGNIFICANCE_THRESHOLD = 50
MIN_NEIGHBOR_ITEMS = 5
MIN_ITEM_RATINGS = 20

### 5.1 Matriz Item × Usuário

No User-Based Collaborative Filtering, cada usuário foi representado pelo conjunto de ratings atribuídos aos itens.

No Item-Based Collaborative Filtering, a perspectiva é invertida: cada filme é representado pelas avaliações recebidas dos diferentes usuários.

A matriz usuário-item construída anteriormente já contém essas informações. Portanto, não é necessário construir uma nova estrutura de dados; para comparar itens, serão utilizadas as colunas da matriz existente.

In [61]:
user_item_matrix.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1660,1662,1663,1664,1670,1672,1673,1675,1676,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,NaN,3.0,NaN,NaN,4.0,1.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
print(
    f"Usuários: {user_item_matrix.shape[0]:,}"
)

print(
    f"Itens:    {user_item_matrix.shape[1]:,}"
)

Usuários: 943
Itens:    1,611


### 5.2 Similaridade entre Itens

A similaridade entre dois filmes será calculada considerando exclusivamente os usuários que avaliaram ambos.

Será utilizada a correlação de Pearson para medir a relação entre os padrões de avaliação dos dois itens.

Também será exigido um número mínimo de usuários em comum para evitar que similaridades sejam calculadas a partir de evidências insuficientes.

In [64]:
def compute_item_similarity(
    user_item_matrix,
    min_common_users=3
):
    item_ids = user_item_matrix.columns
    n_items = len(item_ids)

    similarity = np.zeros(
        (n_items, n_items),
        dtype=float
    )

    overlap = np.zeros(
        (n_items, n_items),
        dtype=int
    )

    for i in range(n_items):
        ratings_i = user_item_matrix.iloc[:, i]

        for j in range(i + 1, n_items):
            ratings_j = user_item_matrix.iloc[:, j]

            # Usuários que avaliaram os dois itens
            common = (
                ratings_i.notna()
                & ratings_j.notna()
            )

            n_common = common.sum()

            overlap[i, j] = n_common
            overlap[j, i] = n_common

            if n_common < min_common_users:
                continue

            x = ratings_i[common].values
            y = ratings_j[common].values

            if np.std(x) == 0 or np.std(y) == 0:
                sim = 0.0
            else:
                sim = np.corrcoef(x, y)[0, 1]

            similarity[i, j] = sim
            similarity[j, i] = sim

    similarity_df = pd.DataFrame(
        similarity,
        index=item_ids,
        columns=item_ids
    )

    overlap_df = pd.DataFrame(
        overlap,
        index=item_ids,
        columns=item_ids
    )

    return similarity_df, overlap_df

In [65]:
item_similarity_pearson, item_overlap = (
    compute_item_similarity(
        user_item_matrix,
        min_common_users=MIN_COMMON_USERS
    )
)

### 5.3 Sobreposição entre Itens

Além da similaridade, é importante considerar a quantidade de usuários que avaliaram cada par de filmes.

Uma correlação elevada calculada a partir de poucos usuários pode ser menos confiável do que uma correlação ligeiramente menor baseada em um número maior de avaliações compartilhadas.

A matriz de sobreposição registra, para cada par de itens, quantos usuários avaliaram ambos.

Nesta primeira implementação, pares com menos de `MIN_COMMON_USERS` avaliações compartilhadas recebem similaridade igual a zero.

In [66]:
item_overlap.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1660,1662,1663,1664,1670,1672,1673,1675,1676,1681
item_id,,,,,,,,,,,,,,,,,,,,,
1,0,66,52,110,41,11,234,124,166,50,...,1,0,0,0,0,1,1,0,0,1
2,66,0,15,68,23,3,61,43,31,17,...,0,0,0,0,0,1,0,0,0,1
3,52,15,0,32,15,2,49,27,30,11,...,0,0,0,0,0,0,0,0,0,0
4,110,68,32,0,36,6,105,76,78,31,...,0,0,0,0,0,1,0,1,1,1
5,41,23,15,36,0,1,54,28,32,5,...,0,0,0,0,0,0,0,0,0,0


In [67]:
invalid_pairs = (
    (item_overlap < MIN_COMMON_USERS)
    & (item_similarity_pearson != 0)
)

print(
    "Pares abaixo do mínimo com similaridade diferente de zero:",
    invalid_pairs.sum().sum()
)

Pares abaixo do mínimo com similaridade diferente de zero: 0


In [68]:
overlap_values = item_overlap.values

positive_overlaps = overlap_values[
    overlap_values > 0
]

print(f"Média de usuários em comum:   {positive_overlaps.mean():.2f}")
print(f"Mediana de usuários em comum: {np.median(positive_overlaps):.2f}")
print(f"Máximo de usuários em comum:  {positive_overlaps.max()}")

Média de usuários em comum:   8.12
Mediana de usuários em comum: 3.00
Máximo de usuários em comum:  405


### 5.4 Significance Weighting

A correlação de Pearson mede a intensidade da relação entre os padrões de avaliação dos itens, mas não considera diretamente a quantidade de observações utilizadas no cálculo.

Uma correlação elevada baseada em poucos usuários pode ocorrer por acaso e ser significativamente menos confiável que uma correlação semelhante baseada em muitas avaliações compartilhadas.

Para reduzir o impacto dessas relações instáveis, será aplicado **significance weighting**, penalizando similaridades calculadas sobre poucos usuários.

O peso será definido por:

\[
w_{ij} =
\min\left(1,\frac{n_{ij}}{\lambda}\right)
\]

e a similaridade ajustada será:

\[
sim^*_{ij} =
sim_{ij} \cdot w_{ij}
\]

onde \(n_{ij}\) é o número de usuários que avaliaram ambos os itens e \(\lambda\) representa o número de observações a partir do qual a similaridade deixa de ser penalizada.

In [95]:
significance_weight = np.minimum(
    item_overlap / SIGNIFICANCE_THRESHOLD,
    1.0
)

item_similarity_weighted = (
    item_similarity_pearson
    * significance_weight
)

print(
    "Similaridade Pearson máxima:",
    item_similarity_pearson.values.max()
)

print(
    "Similaridade ajustada máxima:",
    item_similarity_weighted.values.max()
)

print(
    "Quantidade de relações alteradas:",
    (
        item_similarity_weighted
        != item_similarity_pearson
    ).sum().sum()
)



Similaridade Pearson máxima: 1.0
Similaridade ajustada máxima: 0.9536462619686605
Quantidade de relações alteradas: 717574


### 5.5 Inspeção das Similaridades

Antes da geração das recomendações, será realizada uma inspeção qualitativa das relações encontradas pelo modelo.

Essa análise permite verificar quais filmes apresentam padrões de avaliação semelhantes e, principalmente, quantos usuários contribuíram para cada correlação.

É importante destacar que similaridade colaborativa não significa necessariamente similaridade de conteúdo. Dois filmes podem apresentar gêneros ou temas diferentes e ainda assim serem considerados semelhantes caso tenham sido avaliados de maneira semelhante pelo mesmo conjunto de usuários.

In [96]:
#Star Wars como exemplo

star_wars = movies[
    movies["title"].str.contains(
        "Star Wars",
        case=False,
        na=False
    )
][["item_id", "title"]]

star_wars_id = star_wars.iloc[0]["item_id"]

similar_items = (
    item_similarity_weighted
    .loc[star_wars_id]
    .drop(index=star_wars_id, errors="ignore")
    .sort_values(ascending=False)
    .head(10)
)

similar_movies = (
    similar_items
    .rename("weighted_similarity")
    .reset_index()
    .merge(
        movies[["item_id", "title"]],
        on="item_id",
        how="left"
    )
)

similar_movies["pearson_similarity"] = (
    similar_movies["item_id"]
    .map(item_similarity_pearson.loc[star_wars_id])
)

similar_movies["common_users"] = (
    similar_movies["item_id"]
    .map(item_overlap.loc[star_wars_id])
)

similar_movies[
    [
        "item_id",
        "title",
        "pearson_similarity",
        "common_users",
        "weighted_similarity"
    ]
]

,item_id,title,pearson_similarity,common_users,weighted_similarity
0,172,"Empire Strikes Back, The (1980)",0.759765,286,0.759765
1,181,Return of the Jedi (1983),0.672575,405,0.672575
2,174,Raiders of the Lost Ark (1981),0.528942,307,0.528942
3,963,Some Folks Call It a Sling Blade (1993),0.748760,30,0.449256
4,249,Austin Powers: International Man of Mystery (1...,0.424790,90,0.424790
5,486,Sabrina (1954),0.527539,38,0.400930
6,194,"Sting, The (1973)",0.378958,168,0.378958
7,614,Giant (1956),0.509248,37,0.376844
8,123,"Frighteners, The (1996)",0.365658,89,0.365658
9,380,Star Trek: Generations (1994),0.364777,68,0.364777


### 5.6 Predição de Scores

Para estimar a preferência de um usuário por um item candidato, serão considerados os itens já avaliados pelo próprio usuário.

Cada rating será ponderado pela similaridade entre o item avaliado e o item candidato.

Dessa forma, filmes mais semelhantes ao candidato exercem maior influência sobre o score estimado.

A predição será calculada utilizando uma média ponderada dos ratings:

\[
\hat r_{ui}
=
\frac{
\sum_{j \in I(u)}
sim(i,j) \cdot r_{uj}
}{
\sum_{j \in I(u)}
|sim(i,j)|
}
\]

onde \(I(u)\) representa os itens já avaliados pelo usuário.

In [112]:
def predict_item_based_scores(
    user_id,
    centered_matrix,
    user_means,
    similarity_matrix,
    train_catalog,
    seen_items,
    eligible_items,
    min_neighbor_items=MIN_NEIGHBOR_ITEMS
):

    # Ratings centralizados dos itens já avaliados pelo usuário
    user_deviations = (
        centered_matrix
        .loc[user_id]
        .dropna()
    )

    # Itens ainda não vistos pelo usuário
    candidates = utils.get_candidate_items(
        user_id=user_id,
        train_catalog=train_catalog,
        seen_items=seen_items
    )

    candidates = [
        item_id
        for item_id in candidates
        if item_id in eligible_items
    ]
    scores = {}

    for item_id in candidates:

        if item_id not in similarity_matrix.index:
            continue

        # Similaridade do candidato com os itens
        # já avaliados pelo usuário
        similarities = (
            similarity_matrix
            .loc[item_id, user_deviations.index]
        )

        # Remove relações sem similaridade válida
        valid = similarities != 0

        if not valid.any():
            continue

        similarities = similarities[valid]

        deviations = user_deviations.loc[
            similarities.index
        ]

        if len(similarities) < min_neighbor_items:
            continue

        denominator = np.abs(similarities).sum()

        if denominator == 0:
            continue

        # Desvio esperado para o item candidato
        weighted_deviation = (
            similarities * deviations
        ).sum() / denominator

        # Retorna para a escala original do usuário
        predicted_rating = (
            user_means.loc[user_id]
            + weighted_deviation
        )

        scores[item_id] = predicted_rating

    return pd.Series(
        scores,
        dtype=float
    ).sort_values(ascending=False)

In [118]:
# Número de avaliações de cada item no conjunto de treino
item_rating_counts = (
    train
    .groupby("item_id")
    .size()
)

# Itens elegíveis para recomendação
eligible_items = set(
    item_rating_counts[
        item_rating_counts >= MIN_ITEM_RATINGS
    ].index
)

In [119]:
#Teste
user_id = evaluation_users[0]

item_scores = predict_item_based_scores(
    user_id=user_id,
    centered_matrix=centered_matrix,
    user_means=user_means,
    similarity_matrix=item_similarity_weighted,
    train_catalog=train_catalog,
    seen_items=seen_items,
    eligible_items=eligible_items,
    min_neighbor_items=MIN_NEIGHBOR_ITEMS
)

item_scores.head(10)

,0
1142,4.164442
919,4.144238
963,4.117938
936,4.114393
116,4.099404
285,4.084758
1009,4.071755
522,4.060976
483,4.056097
100,4.051498


In [105]:
user_id = evaluation_users[0]

item_scores = predict_item_based_scores(
    user_id=user_id,
    centered_matrix=centered_matrix,
    user_means=user_means,
    similarity_matrix=item_similarity_weighted,
    train_catalog=train_catalog,
    seen_items=seen_items,
    min_neighbor_items=MIN_NEIGHBOR_ITEMS
)

item_scores.head(10)

,0
1598,4.960491
1367,4.788675
1238,4.783430
994,4.616115
1405,4.549808
1312,4.543748
1589,4.523618
1448,4.507712
1454,4.496185
1361,4.492147


### 5.6 Recomendações Top-K

Para cada usuário avaliável, serão calculados os scores dos itens ainda não observados no conjunto de treino.

Os itens candidatos serão ordenados pelo rating estimado, e os \(K\) itens com maior score formarão a lista final de recomendações do Item-Based Collaborative Filtering.

In [121]:
item_cf_recommendations = {}

for user_id in evaluation_users:

    scores = predict_item_based_scores(
        user_id=user_id,
        centered_matrix=centered_matrix,
        user_means=user_means,
        similarity_matrix=item_similarity_weighted,
        train_catalog=train_catalog,
        seen_items=seen_items,
        eligible_items=eligible_items,
        min_neighbor_items=MIN_NEIGHBOR_ITEMS
    )

    item_cf_recommendations[user_id] = (
        scores
        .head(K)
        .index
        .tolist()
    )

item_recommendation_lengths = pd.Series({
    user_id: len(items)
    for user_id, items in item_cf_recommendations.items()
})

item_recommendation_lengths.describe()

,0
count,907.0
mean,10.0
std,0.0
min,10.0
25%,10.0
50%,10.0
75%,10.0
max,10.0


In [122]:
print(
    f"Usuários com menos de {K} recomendações:",
    (item_recommendation_lengths < K).sum()
)

print(
    "Percentual:",
    f"{(item_recommendation_lengths < K).mean():.2%}"
)

Usuários com menos de 10 recomendações: 0
Percentual: 0.00%


### 5.7 Avaliação

O Item-Based Collaborative Filtering será avaliado utilizando o mesmo ground truth e as mesmas métricas dos modelos anteriores.

A manutenção do protocolo experimental permite comparar diretamente o desempenho das estratégias User-Based e Item-Based, isolando a mudança na abordagem de Collaborative Filtering.

In [123]:
item_cf_results, item_cf_summary = (
    utils.evaluate_recommendations(
        recommendations=item_cf_recommendations,
        ground_truth=ground_truth,
        k=K
    )
)

item_cf_summary

{'Precision@10': np.float64(0.020507166482910698),
 'Recall@10': np.float64(0.015441376161067454),
 'NDCG@10': np.float64(0.021425876328077884),
 'n_users': 907}

In [124]:
comparison2 = utils.add_model_result(
    results_df=comparison,
    model_name="Item-Based CF",
    summary=item_cf_summary,
    k=K
)

comparison2

,model,Precision@10,Recall@10,NDCG@10,n_users
0,Random,0.007387,0.006949,0.008209,907
1,Popularity,0.077508,0.078646,0.098883,907
2,Content-Based,0.018964,0.018318,0.025018,907
3,User-Based CF,0.013671,0.011940,0.015679,907
4,Item-Based CF,0.020507,0.015441,0.021426,907


### 5.9 Análise dos Resultados

O Item-Based Collaborative Filtering foi construído utilizando a correlação de Pearson entre os padrões de avaliação dos itens, considerando apenas usuários que avaliaram ambos os filmes.

Como a matriz usuário-item do MovieLens 100K é esparsa, similaridades calculadas a partir de poucas avaliações compartilhadas podem apresentar valores elevados mesmo quando sustentadas por pouca evidência. Para reduzir esse efeito, foram utilizados mecanismos adicionais de controle da confiabilidade das recomendações.

Os parâmetros adotados foram:

- `MIN_COMMON_USERS = 3`: quantidade mínima de usuários em comum necessária para calcular a similaridade entre dois itens;
- `SIGNIFICANCE_THRESHOLD = 50`: penaliza progressivamente similaridades calculadas com poucos usuários compartilhados;
- `MIN_NEIGHBOR_ITEMS = 5`: exige que pelo menos cinco itens do histórico do usuário contribuam para a estimativa de um candidato;
- `MIN_ITEM_RATINGS = 20`: restringe os candidatos a itens com pelo menos 20 avaliações no conjunto de treino.

O valor de `MIN_ITEM_RATINGS` corresponde aproximadamente à mediana da distribuição de avaliações por item observada no conjunto de treino.

A configuração final apresentou os seguintes resultados:

| Métrica | Resultado |
|---|---:|
| Precision@10 | 0.02051 |
| Recall@10 | 0.01544 |
| NDCG@10 | 0.02143 |

Comparado aos demais modelos, o Item-Based CF apresentou a maior Precision@10 entre os modelos personalizados avaliados, superando o Content-Based e o User-Based CF nessa métrica.

O Content-Based, entretanto, apresentou Recall@10 e NDCG@10 superiores, indicando melhor capacidade de recuperar itens relevantes e posicioná-los nas primeiras posições do ranking.

O Popularity Baseline permaneceu significativamente superior nas três métricas, evidenciando a força do sinal de popularidade neste conjunto de dados e protocolo experimental.

Uma limitação importante desta implementação é a utilização de `MIN_ITEM_RATINGS = 20`. Esse critério reduz a exposição a itens com poucas avaliações e aumenta a estabilidade das relações colaborativas, mas também restringe o catálogo disponível para recomendação.

Consequentemente, a comparação com modelos que utilizam todo o catálogo não é completamente controlada. Uma avaliação futura poderá aplicar o mesmo universo de candidatos a todos os modelos e incluir métricas de cobertura e diversidade, permitindo analisar o trade-off entre qualidade do ranking e exploração da cauda longa.

## 6. Conclusões

Neste experimento foram implementadas e avaliadas duas abordagens clássicas de Collaborative Filtering: **User-Based CF** e **Item-Based CF**.

O User-Based CF utiliza relações de similaridade entre usuários para estimar preferências, enquanto o Item-Based CF utiliza relações entre itens construídas a partir dos padrões de avaliação dos usuários.

Os resultados obtidos foram:

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
|---|---:|---:|---:|
| Random | 0.00739 | 0.00695 | 0.00821 |
| Popularity | **0.07751** | **0.07865** | **0.09888** |
| Content-Based | 0.01896 | 0.01832 | 0.02502 |
| User-Based CF | 0.01367 | 0.01194 | 0.01568 |
| Item-Based CF | 0.02051 | 0.01544 | 0.02143 |

O **Popularity Baseline apresentou o melhor desempenho nas três métricas**, demonstrando a força do sinal de popularidade no MovieLens 100K e reforçando a importância de comparar modelos mais sofisticados com baselines simples.

Entre os modelos personalizados, não houve um vencedor absoluto. O **Item-Based CF apresentou a maior Precision@10**, enquanto o **Content-Based apresentou os melhores resultados de Recall@10 e NDCG@10**.

Os experimentos com Collaborative Filtering também evidenciaram desafios importantes associados à esparsidade da matriz usuário-item. A confiabilidade das similaridades depende da quantidade de avaliações compartilhadas, e relações calculadas sobre poucas observações podem ser instáveis.

No Item-Based CF, foram utilizados mecanismos de controle como **significance weighting**, número mínimo de itens contribuindo para a predição e quantidade mínima de avaliações por item. Essas estratégias aumentam a confiabilidade das estimativas, mas introduzem um trade-off entre estabilidade das recomendações e cobertura do catálogo.

Em particular, o filtro de quantidade mínima de avaliações restringe o universo de candidatos do Item-Based CF. Dessa forma, a comparação direta com modelos que utilizam todo o catálogo deve ser interpretada com cautela. Uma avaliação futura poderá utilizar o mesmo conjunto de candidatos para todos os modelos e incorporar métricas de cobertura, diversidade e exploração da cauda longa.

Os hiperparâmetros utilizados nesta implementação foram definidos com finalidade experimental e não devem ser considerados valores ótimos. Uma etapa posterior poderá utilizar um conjunto de validação para realizar a seleção sistemática desses parâmetros, preservando o conjunto de teste exclusivamente para a avaliação final.

Como próxima evolução do projeto, serão avaliados métodos de **Matrix Factorization**, capazes de representar usuários e itens em um espaço latente e oferecer uma abordagem diferente dos métodos de vizinhança utilizados neste notebook.